# Notebook 03 — `refit_and_forecast` mode: PySR refit + Fisher forecast

## Prerequisites

1. **PYTHONPATH** — `src/` of this repo + `lya_emulator_full`:
   ```
   export PYTHONPATH=/path/to/lya_emulator_full:<repo_root>/src
   ```
2. **`data/kodiaq_gp/`** — the GP emulator basedir (committed to the repo).
3. **`data/single_z_1pvar/`** — regenerated 1pvar HDF5s (raw P_F, all 13
   z-bins and 11 parameters). Generate once with:
   ```bash
   python scripts/regen_1pvar.py \
       --basedir data/kodiaq_gp \
       --output  data/single_z_1pvar
   ```
   Runtime: ~10–15 minutes (loads emulators + sweeps 11 params × 13 z-bins).
4. **Julia + PySR** — `refit_and_forecast` calls PySR, which requires Julia.
   The following environment variables must point at a pre-built Julia depot:
   ```bash
   export PYTHON_JULIAPKG_PROJECT=/path/to/juliapkg/project
   export JULIA_DEPOT_PATH=/path/to/julia_depot
   ```
   On Greatlakes these are set in the Lya forecast environment module.
   Without them, `import pysr` will attempt to download/install Julia at
   runtime (slow or blocked on compute nodes).

## What this notebook does

Runs the complete single-z refit + forecast loop at z = 3.6:

1. Configure `refit_and_forecast` mode.
2. (Optional) Show how to run `regen_1pvar` from Python.
3. Call `run(cfg)` which loops `refit_one_param_single_z` over the 11
   parameters in-process, writes `pareto_{param}.csv`, then runs the three
   Fisher forecasts.
4. Inspect the emitted Pareto CSVs and the σ ladder.
5. Point at `run_batch.py` for the full 13-z-bin SLURM workflow.

**Expected runtime** (single z-bin, in-process, `niterations=50`):
~20–40 minutes depending on the node. For the full 13-z-bin run, use
`run_batch.py --phase submit` to parallelize on SLURM.

### What `refit_one_param_single_z` does for each parameter

1. Builds LF and HF GP emulators from `cfg.gp.basedir`.
2. Calls `refit_1d_pysr.refit_1d_for_param` (the inline 1pvar path):
   sweeps `(θ_i, k_grid)` at LF resolution (r=0.4) and HF (r=0.8),
   normalizes flux, hands to PySR.
3. Writes `<output_dir>/refit/z{z}/pareto_{param}.csv` — the full Pareto
   front of candidate equations.
4. Returns a `Refit1DResult` (best equation + per-k normalization).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../").resolve()
LYA_EMU_ROOT = Path("/home/mfho/student_projects/lya_emulator_full")

for p in [str(REPO_ROOT / "src"), str(LYA_EMU_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("sys.path configured.")

## 1. (Optional) Regenerate per-parameter training data

Skip this cell if `data/single_z_1pvar/` already exists.

The script `regen_1pvar.py` can also be called from Python directly via
`training_data.regenerate_param` + `write_1pvar_hdf5`.
The shell command is simpler for the full 11-parameter run:

In [ ]:
import subprocess, os

data_dir = REPO_ROOT / "data" / "single_z_1pvar"

if data_dir.exists() and any(data_dir.iterdir()):
    print(f"data/single_z_1pvar/ already populated ({len(list(data_dir.iterdir()))} files).")
    print("Skipping regen_1pvar.")
else:
    print("Running regen_1pvar.py (takes ~10-15 min) ...")
    env = {**os.environ, "PYTHONPATH": ":".join(sys.path)}
    proc = subprocess.run(
        [sys.executable, str(REPO_ROOT / "scripts" / "regen_1pvar.py"),
         "--basedir", str(REPO_ROOT / "data" / "kodiaq_gp"),
         "--output",  str(data_dir)],
        capture_output=True, text=True, env=env,
    )
    print(proc.stdout[-3000:])   # last 3k chars
    if proc.returncode != 0:
        print("STDERR:", proc.stderr[-2000:])

## 2. Configure `refit_and_forecast`

The `pysr:` block controls the symbolic regression budget. Defaults are
suitable for a production run. For quick testing, reduce `niterations`.

In [ ]:
from priya_forecast.single_z.config import (
    PipelineConfig, DataConfig, GPConfig, KRange,
    PySRConfig, ParetoCSVsConfig,
)

cfg = PipelineConfig(
    mode="refit_and_forecast",
    redshift=3.6,
    output_dir="/tmp/nb03_refit_forecast/",
    data=DataConfig(source="kodiaq", conservative=True, mock_data="gp"),
    gp=GPConfig(basedir=str(REPO_ROOT / "data/kodiaq_gp")),
    k_range=KRange(min=0.001, max=0.04),
    combine="additive",
    pick="best_loss",
    pysr=PySRConfig(
        smart_kwargs=True,
        use_anova_loss=False,   # OFF: clean baseline; ANOVA is principled but marginal
        niterations=50,         # reduce to 5-10 for a quick smoke-test
        populations=24,
        procs=4,
        maxsize=20,
        seed=0,
    ),
    pareto_csvs=ParetoCSVsConfig(source="from_refit"),
)
cfg.validate()
print(f"mode        : {cfg.mode}")
print(f"redshift    : {cfg.redshift}")
print(f"niterations : {cfg.pysr.niterations}")
print(f"smart_kwargs: {cfg.pysr.smart_kwargs}")
print(f"output_dir  : {cfg.output_dir}")

## 3. Run the full refit + forecast pipeline

`run_refit_and_forecast` (called by `run(cfg)`) does:
1. Build LF + HF GP emulators (`GPModel(fidelity="lf"/"hf")`).
2. Loop `refit.refit_one_param_single_z` over `cfg.parameters`:
   each call runs `refit_1d_for_param` (inline 1pvar path inside
   `refit_1d_pysr.py`) and writes `pareto_{param}.csv`.
3. Pass the returned `Refit1DResult` objects to `forecast.run_three_fisher`
   (same Fisher machinery as `forecast_only`).
4. Write outputs: `forecast_table.txt`, `scorecard.md`, `corner.png`,
   `fisher_{GP,perfect_1D,PySR}.npz`.

The Pareto CSVs land in `<output_dir>/refit/z3.6/pareto_{param}.csv`.

In [ ]:
from priya_forecast.single_z.pipeline import run

# This will take ~20-40 minutes for all 11 parameters at niterations=50.
# Reduce niterations in cfg above for a quick smoke-test.
result = run(cfg)

print("Result keys:", list(result.keys()))
print("refit_dir  :", result.get("refit_dir"))

## 4. Inspect emitted Pareto CSVs

In [ ]:
from pathlib import Path

refit_dir = Path(result["refit_dir"])
csvs = sorted(refit_dir.glob("pareto_*.csv"))
print(f"Pareto CSVs written to {refit_dir}:")
for csv in csvs:
    print(f"  {csv.name}  ({csv.stat().st_size} bytes)")

In [ ]:
# Inspect one Pareto front — the equation PySR discovered for 'ns'
from priya_forecast.models.pysr_model import load_pareto_csv

ns_csv = refit_dir / "pareto_ns.csv"
if ns_csv.exists():
    df = load_pareto_csv(ns_csv)
    print("Pareto front for 'ns' (all complexity levels):")
    print(df[["Complexity", "Loss", "Equation"]].to_string(index=False))
else:
    print("ns CSV not yet written — run cell 3 first.")

## 5. Inspect the σ ladder

In [ ]:
import numpy as np

sigmas = result["sigmas"]    # dict: "GP" | "perfect_1D" | "PySR" → ndarray
params = cfg.parameters

print(f"{'param':<12s}  {'σ_GP':>10s}  {'σ_perf1D':>10s}  {'σ_PySR':>10s}  "
      f"{'PySR/GP':>8s}")
print("-" * 60)
for i, name in enumerate(params):
    sg = sigmas["GP"][i]
    sp = sigmas["perfect_1D"][i]
    sy = sigmas["PySR"][i]
    print(f"{name:<12s}  {sg:>10.4g}  {sp:>10.4g}  {sy:>10.4g}  {sy/sg:>8.3f}")

In [ ]:
print(open(result["table_path"]).read())

In [ ]:
print(open(result["scorecard_path"]).read())

## 6. Running a single (param, z) refit

You can refit just one parameter without running the full pipeline:

In [ ]:
import numpy as np
from priya_forecast.models.gp_model import GPModel
from priya_forecast.single_z import refit as _refit
from priya_forecast.parameters import PARAMS_11D

k_grid = _refit.kodiaq_k_grid(cfg.k_range.min, cfg.k_range.max, 48)
fid = np.array([p.fid for p in PARAMS_11D], dtype=float)

gp_lf = GPModel(basedir=cfg.gp.basedir, fidelity="lf", kf=k_grid)
gp_hf = GPModel(basedir=cfg.gp.basedir, fidelity="hf", kf=k_grid)

refit_dir_single = Path("/tmp/nb03_single_param/")

# Refit 'ns' only (fast smoke-test with niterations=5):
cfg_quick = cfg
cfg_quick.pysr.niterations = 5

result_ns = _refit.refit_one_param_single_z(
    param_name="ns",
    z=3.6,
    cfg=cfg_quick,
    gp_lf=gp_lf,
    gp_hf=gp_hf,
    k_grid=k_grid,
    out_dir=refit_dir_single,
)

print(f"Best equation for 'ns' (complexity={result_ns.pareto_complexity}, "
      f"loss={result_ns.pareto_loss:.4g}):")
print(f"  {result_ns.equation_str}")

## 7. All 13 z-bins on SLURM (recommended for production)

Running all 11 parameters × 13 z-bins in-process would take ~5 hours.
Instead, use `run_batch.py` to fan over z-bins and submit SLURM array
jobs (one job per z-bin, 11 tasks per job = 11 parameters):

```bash
# 1. Regenerate training data (once, ~15 min)
python scripts/regen_1pvar.py \
    --basedir data/kodiaq_gp \
    --output  data/single_z_1pvar

# 2. Submit 13 SLURM array jobs (--array=0-10, one task per parameter)
python scripts/run_batch.py \
    --config configs/single_z/example.yaml \
    --mode   refit_and_forecast \
    --phase  submit
# Prints job IDs. Wait for them to finish.

# 3. Collect + forecast + aggregate
python scripts/run_batch.py \
    --config configs/single_z/example.yaml \
    --mode   refit_and_forecast \
    --phase  collect
# Checks all 11 × 13 = 143 pareto_*.csv files are present,
# then runs forecast_only per z-bin, then calls aggregate_z.
```

Each SLURM task is one call to `scripts/refit_one_param_single_z.py`:

```bash
python scripts/refit_one_param_single_z.py \
    --param ns --z 3.6 \
    --basedir data/kodiaq_gp \
    --output-dir results/single_z_run
```

### Across-z aggregation

After all z-bins are forecast, view the σ(z) trend:

```bash
python scripts/aggregate_z.py --base results/single_z_run
# Writes results/single_z_run/aggregate/sigma_vs_z.png
#        results/single_z_run/aggregate/sigma_table.md
```